<a href="https://colab.research.google.com/github/eshikanahata/DC-Mini-Project/blob/Shashank/task3_RAG.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# The RAG Pipeline (Chunking -> Vector Store -> Generation)

Objective:  
You will learn why "how you read" data (Chunking) matters as much as "what you read," and you will build a Retrieval Augmented Generation (RAG) pipeline.  
  



# Section 0: Setup & Prerequisites

Install the necessary libraries. You will likely need langchain, langchain-community, chromadb, and an embedding provider.

In [1]:
# INITIAL SETUP (RUN THIS ONCE)

# 1. Install dependencies with specific versions to avoid conflicts
!pip install -q -U \
  torch \
  transformers \
  sentence-transformers \
  accelerate \
  bitsandbytes \
  langchain \
  langchain-community \
  chromadb \
  pysqlite3-binary

# 2. Fix Colab's SQLite version issue (Must happen before importing chromadb)
__import__('pysqlite3')
import sys
sys.modules['sqlite3'] = sys.modules.pop('pysqlite3')

# 3. Check if GPU is available
import torch
if not torch.cuda.is_available():
    print("WARNING: You are running on CPU. Go to Runtime -> Change runtime type -> T4 GPU")
else:
    print(f" GPU Detected: {torch.cuda.get_device_name(0)}")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.4/10.4 MB 115.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 15.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 77.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.5/21.5 MB 80.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.0/5.0 MB 124.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 34.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 108.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 77.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.1/17.1 MB 102.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 9.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 132.6/132.6 kB 17.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.4/66.

In [2]:
# TODO: Import any other necessary libraries here
!pip install -q langchain-text-splitters

# Section 1: Chunking Experiment

LLMs have context windows. We must slice our data. But if you slice a sentence in half, the meaning might be lost. Let's prove this.

In [3]:
# Load a text file of your choice

filename = "/content/llm_wikipedia.txt"

def load_data(path):
    try:
        with open(path, 'r', encoding='utf-8') as f:
            return f.read()
    except FileNotFoundError:
        return "Error. File not found"

raw_text = load_data(filename)
print(f"Loaded {len(raw_text)} characters.")

Loaded 65063 characters.


Implement a splitter that strictly cuts text every x characters, regardless of sentence boundaries.

In [4]:
def naive_splitter(text, chunk_size=500):
    """
    Splits text strictly by character count.
    Returns: List[str]
    """
    # TODO: Implement strictly fixed-size splitting WITHOUT using a library (use pure Python)
    text_chunks = []
    texts = text
    while chunk_size < len(texts):
        text_chunks.append(texts[:chunk_size])
        texts = texts[chunk_size:]
    else:
        text_chunks.append(texts)
    return text_chunks

naive_chunks = naive_splitter(raw_text)

Use a library (like LangChain) to split by "separators" (Paragraphs $\rightarrow$ Sentences $\rightarrow$ Words) to preserve meaning.

In [5]:
# TODO: Initialize a RecursiveCharacterTextSplitter
# Docs: Look up LangChain Text Splitters
# Constraints: Chunk size x, Chunk Overlap t

from langchain_text_splitters import RecursiveCharacterTextSplitter

specific_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50,
    separators=["\n\n", "\n", " ", ""] # The order of priority for splitting
)

semantic_chunks = specific_splitter.split_text(raw_text)

Find a specific example where the Naive splitter broke a sentence in half, rendering it meaningless, but the Semantic splitter kept it intact

In [6]:
def find_broken_context(naive_list, semantic_list):
    """
    Print a side-by-side comparison of a specific segment where
    Naive failed and Semantic succeeded.
    """
    # TODO: Write logic to compare chunks or manually inspect to find the error case.

    # consider the first chunks in both naive and semantic
    naive_cut = naive_list[0]
    semantic_cut = semantic_list[0]

    print(f"End of chunk 1 (Naive splitter):")
    print(f"...{naive_cut[-30:]}")

    print(f"\nStart of chunk 2 (Naive splitter):")
    print(f"{naive_list[1][:20]}...")

    print("\n" + "="*40)

    print(f"\nEnd of chunk 1 (Semantic splitter):")
    print(f"...{semantic_cut[-30:]}")

    print(f"\nStart of chunk 2 (Semantic splitter):")
    print(f"{semantic_list[1][:20]}...")

find_broken_context(naive_chunks, semantic_chunks)

End of chunk 1 (Naive splitter):
...ctive power regarding syntax, 

Start of chunk 2 (Naive splitter):
semantics, and ontol...


End of chunk 1 (Semantic splitter):
...Large Language Model

Start of chunk 2 (Semantic splitter):
A large language mod...


In a text cell below, explain why the overlap parameter in the recursive splitter is essential for retrieval tasks.

If we split chunks with no overlap, the semantic meaning/relationship between consecutive chunks might be lost. This can cause the model to hallucinate if it can't bridge the context between 2 chunks.

By introducing the overlap (like 50 characters) we can give the model some context of the previous chunk, so that it doesn't get totally confused. This greatly reduces hallucinations

# Section 2: Vector Storage

We will convert our semantic_chunks into vector embeddings and store them.

Initialize Embeddings & DB:
- You may use OpenAI Embeddings (if you have a key) or HuggingFace (all-MiniLM-L6-v2) for a free, local alternative.
- Use ChromaDB or FAISS as your store.

In [7]:
# TODO: Initialize your Embedding Model
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma

model_name = "sentence-transformers/all-MiniLM-L6-v2"
model_kwargs = {'device': 'cuda'}
encode_kwargs = {'normalize_embeddings': False}
embedding_function = HuggingFaceEmbeddings(
    model_name=model_name,
    model_kwargs=model_kwargs,
    encode_kwargs=encode_kwargs
)

# TODO: Create a Vector Store from your `semantic_chunks`
# Hint: Look for `.from_texts` or `.from_documents` in the LangChain/Chroma documentation.

vector_db = Chroma.from_texts(
    texts=semantic_chunks,
    embedding=embedding_function,
    persist_directory="./chroma_db"
)
print("Vector Store successfully created.")

/tmp/ipython-input-315/1507274630.py:8: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embedding_function = HuggingFaceEmbeddings(
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Vector Store successfully created.


# Section 3: The MVP (Retrieval Loop)

We have the brain (LLM) and the memory (Vector DB). Now we need to wire them together.

Create a function that takes a user query, converts it to a vector, and finds the top 3 most relevant chunks from your database.

In [8]:
def retrieve_context(query, k=3):
    """
    Args:
        query (str): The user's question
        k (int): Number of chunks to retrieve
    Returns:
        List[str]: The top k context chunks
    """
    # TODO: Use your vector_db to perform a similarity search
    matching_docs = vector_db.similarity_search(query, k=k)

    context_chunks = [doc.page_content for doc in matching_docs]
    return context_chunks

# Test it
test_query = ""
context_results = retrieve_context(test_query)
print(f"Retrieved {len(context_results)} chunks.")

Retrieved 3 chunks.


Construct the final prompt. You must inject the retrieved context into the system prompt so the LLM answers only based on that data.

In [24]:
# TODO: Initialize your LLM (OpenAI, Anthropic, or a local Llama via Ollama/HuggingFace)
from langchain_community.llms import HuggingFacePipeline
from transformers import pipeline

pipe = pipeline(
    task="text-generation",
    model="TinyLlama/TinyLlama-1.1B-Chat-v1.0",
    max_new_tokens=256,
    device=0,
    return_full_text=False
)

llm = HuggingFacePipeline(pipeline=pipe)


def generate_answer(query):
    # 1. Retrieve context
    context_chunks = retrieve_context(query)
    context_str = "\n\n".join(context_chunks)

    # 2. Construct the Prompt
    # Constraint: The prompt must instruct the LLM to say "I don't know" if the info isn't in the chunks.
    prompt = f"""Context: {context_str}

Question: {query}
Answer (If the answer is not in the context, reply exactly with "I don't know"): """

    # 3. Pass to LLM and return response
    response = llm.invoke(prompt)
    return response

# Final Test
answer = generate_answer("what is a large language model? what is it used for?")
print(answer)

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]


A large language model (LLM) is a state-of-the-art machine learning model that is used for natural language processing (NLP) tasks. It is a pre-trained language model that has been trained on vast amounts of text data to learn complex language representations. LLMs are used in a variety of tasks, including text generation, question answering, and natural language inference. They are particularly effective at handling unstructured and domain-specific text, making them highly suitable for applications where precision and efficiency are critical. While LLMs have shown great promise in a range of applications, they have significant limitations that must be addressed to make them a viable solution for high-stakes applications.


# The End (of task 3)